# Find chronological splits for SciQuality labels

The same chronological logic is applied to all three stages. Each candidate has a complete 24-hour NF-only gap, a holdout between 3% and 15%, and sufficient NF and FL images in both training and holdout.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")

# Find the repository root.
ROOT = Path.cwd().resolve()
while not (ROOT / "data_labeling").is_dir():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Repository root could not be found.")
    ROOT = ROOT.parent

LABEL_FILE = ROOT / "data_labeling/data_labels/sciquality/labels_sciquality_2010_2026_binary.csv"
assert LABEL_FILE.exists(), f"File not found: {LABEL_FILE}"

In [ ]:
# Load labels and recover timestamps from image paths.
data = pd.read_csv(LABEL_FILE)
timestamp_text = data["label"].str.extract(r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})")[0]
data["timestamp"] = pd.to_datetime(timestamp_text, format="%Y.%m.%d_%H.%M.%S", errors="coerce")
data["goes_class"] = pd.to_numeric(data["goes_class"], errors="coerce")
data = data.sort_values("timestamp").reset_index(drop=True)

assert data["label"].duplicated().sum() == 0
assert data[["timestamp", "goes_class"]].isna().sum().sum() == 0
assert set(data["goes_class"].unique()) <= {0, 1}
print(f"Images loaded: {len(data):,}")

## Chronological periods and major gaps

In [ ]:
# Keep the previous complete-stage boundaries for a controlled comparison.
BLOCK_LIMITS = {
    "Stage 1": (pd.Timestamp("2010-12-06 07:00"), pd.Timestamp("2017-12-31 23:00")),
    "Stage 2": (pd.Timestamp("2018-01-02 00:00"), pd.Timestamp("2023-06-29 22:00")),
    "Stage 3": (pd.Timestamp("2023-07-01 00:00"), pd.Timestamp("2024-06-29 10:00")),
    "Validation": (pd.Timestamp("2024-07-01 00:00"), pd.Timestamp("2025-06-30 23:00")),
    "Test": (pd.Timestamp("2025-07-02 00:00"), pd.Timestamp("2026-07-31 23:00")),
}
MAJOR_GAPS = {
    "Stage 1 → Stage 2": (pd.Timestamp("2018-01-01 00:00"), pd.Timestamp("2018-01-01 23:00")),
    "Stage 2 → Stage 3": (pd.Timestamp("2023-06-29 23:00"), pd.Timestamp("2023-06-30 22:00")),
    "Stage 3 → Validation": (pd.Timestamp("2024-06-29 11:00"), pd.Timestamp("2024-06-30 10:00")),
    "Validation → Test": (pd.Timestamp("2025-07-01 00:00"), pd.Timestamp("2025-07-01 23:00")),
}

blocks = {name: data[data["timestamp"].between(start, end)].copy() for name, (start, end) in BLOCK_LIMITS.items()}

In [ ]:
def summarize(name, frame):
    total = len(frame)
    fl = int(frame["goes_class"].sum())
    return {"dataset": name, "start": frame["timestamp"].min(), "end": frame["timestamp"].max(),
            "samples": total, "NF": total - fl, "FL": fl,
            "FL_percent": 100 * fl / total if total else np.nan}

display(pd.DataFrame([summarize(name, frame) for name, frame in blocks.items()]))

In [ ]:
def check_gap(name, start, end):
    gap = data[data["timestamp"].between(start, end)]
    expected = pd.date_range(start, end, freq="h")
    missing = expected.difference(gap["timestamp"])
    fl = int(gap["goes_class"].sum())
    return {"boundary": name, "start": start, "end": end, "samples": len(gap),
            "FL": fl, "missing_hours": len(missing),
            "valid_gap": len(gap) == 24 and fl == 0 and len(missing) == 0}

major_gap_summary = pd.DataFrame([check_gap(name, start, end) for name, (start, end) in MAJOR_GAPS.items()])
display(major_gap_summary)
assert major_gap_summary["valid_gap"].all()
print("All major boundaries have complete 24-hour NF-only gaps.")

## Check the previous internal gaps

In [ ]:
PREVIOUS_INTERNAL_GAPS = {
    "Stage 1": (pd.Timestamp("2017-01-06 04:00"), pd.Timestamp("2017-01-07 03:00")),
    "Stage 2": (pd.Timestamp("2023-03-08 23:00"), pd.Timestamp("2023-03-09 22:00")),
    "Stage 3": (pd.Timestamp("2024-05-06 06:00"), pd.Timestamp("2024-05-07 05:00")),
}
previous_gap_summary = pd.DataFrame([check_gap(name, start, end) for name, (start, end) in PREVIOUS_INTERNAL_GAPS.items()])
display(previous_gap_summary)

## Search for valid train/holdout candidates

In [ ]:
def find_holdout_candidates(stage_data, target_percent=15, minimum_percent=3,
                            minimum_train_fl=500, minimum_train_nf=500,
                            minimum_holdout_fl=200, minimum_holdout_nf=200):
    """Find holdouts after complete 24-hour NF gaps using the previous logic."""
    stage_data = stage_data.sort_values("timestamp").reset_index(drop=True)
    hourly_index = pd.date_range(stage_data["timestamp"].min(), stage_data["timestamp"].max(), freq="h")
    hourly = stage_data.set_index("timestamp")["goes_class"].reindex(hourly_index)
    gap_ends = hourly.index[(hourly.rolling(24).count() == 24) & (hourly.rolling(24).sum() == 0)]
    stage_fl_percent = 100 * stage_data["goes_class"].mean()
    candidates = []

    for gap_end in gap_ends:
        gap_start = gap_end - pd.Timedelta(hours=23)
        holdout_start = gap_end + pd.Timedelta(hours=1)
        training = stage_data[stage_data["timestamp"] < gap_start]
        holdout = stage_data[stage_data["timestamp"] >= holdout_start]
        if training.empty or holdout.empty:
            continue

        train_fl = int(training["goes_class"].sum()); train_nf = len(training) - train_fl
        holdout_fl = int(holdout["goes_class"].sum()); holdout_nf = len(holdout) - holdout_fl
        holdout_percent = 100 * len(holdout) / (len(training) + len(holdout))
        enough_data = (train_fl >= minimum_train_fl and train_nf >= minimum_train_nf
                       and holdout_fl >= minimum_holdout_fl and holdout_nf >= minimum_holdout_nf
                       and holdout_fl < train_fl)
        if not enough_data or not minimum_percent <= holdout_percent <= target_percent:
            continue

        holdout_fl_percent = 100 * holdout_fl / len(holdout)
        candidates.append({"gap_start": gap_start, "gap_end": gap_end, "holdout_start": holdout_start,
            "train_samples": len(training), "train_FL": train_fl, "train_NF": train_nf,
            "holdout_samples": len(holdout), "holdout_FL": holdout_fl, "holdout_NF": holdout_nf,
            "holdout_percent": holdout_percent, "holdout_FL_percent": holdout_fl_percent,
            "stage_FL_percent": stage_fl_percent,
            "class_shift_points": abs(holdout_fl_percent - stage_fl_percent),
            "size_error_points": abs(target_percent - holdout_percent)})

    candidates = pd.DataFrame(candidates)
    if candidates.empty:
        raise ValueError("No holdout satisfies the requirements.")
    return candidates.sort_values(["size_error_points", "class_shift_points", "gap_start"]).reset_index(drop=True)

In [ ]:
# Display the ten best candidates for all stages.
stage_sets = {name: blocks[name] for name in ["Stage 1", "Stage 2", "Stage 3"]}
candidate_tables = {}
for stage_name, stage_data in stage_sets.items():
    candidate_tables[stage_name] = find_holdout_candidates(stage_data)
    print(f"\n{stage_name}: ten best candidates")
    display(candidate_tables[stage_name].head(10))

The first row is closest to a 15% holdout after all safety and sample-count rules are applied. Review all ten rows before accepting a split because temporal class shift is expected.

In [ ]:
# Select the first candidate provisionally. No CSV files are written.
selected_splits = {}
for stage_name, stage_data in stage_sets.items():
    selected = candidate_tables[stage_name].iloc[0]
    selected_splits[stage_name] = {
        "training": stage_data[stage_data["timestamp"] < selected["gap_start"]].copy(),
        "gap": stage_data[stage_data["timestamp"].between(selected["gap_start"], selected["gap_end"])].copy(),
        "holdout": stage_data[stage_data["timestamp"] >= selected["holdout_start"]].copy(),
    }

rows = []
for stage_name, split in selected_splits.items():
    for role in ["training", "gap", "holdout"]:
        row = summarize(f"{stage_name} {role}", split[role]); row["role"] = role.title(); rows.append(row)
split_summary = pd.DataFrame(rows)
display(split_summary[["dataset", "role", "start", "end", "samples", "NF", "FL", "FL_percent"]])

In [ ]:
# Plot natural train and holdout distributions.
plot_data = split_summary[split_summary["role"].isin(["Training", "Holdout"])].set_index("dataset")
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_data[["NF", "FL"]].plot(kind="bar", stacked=True, color=["steelblue", "darkorange"], ax=axes[0])
plot_data["FL_percent"].plot(kind="bar", color="darkorange", ax=axes[1])
axes[0].set_title("Natural class distribution"); axes[1].set_title("FL percentage")
for axis in axes:
    axis.set_xlabel(""); axis.tick_params(axis="x", rotation=35)
plt.tight_layout(); plt.show()

In [ ]:
# Future periods remain natural and are not used to find internal gaps.
future_summary = pd.DataFrame([summarize("Validation", blocks["Validation"]), summarize("Test", blocks["Test"])])
display(future_summary)